# SatQuery: a small VLM for free-form questions, trained on a free Kaggle GPU

Fine-tunes **HuggingFaceTB/SmolVLM-256M-Instruct** (small enough for the free CPU server) with LoRA on VRSBench's open questions and
captions, using the exact prompts the app sends. Then scores the untrained and the adapted model on the same
held-out evaluation rows (open-VQA accuracy, caption BLEU-4 / ROUGE-L) and times answers on the CPU.

**Settings:** Accelerator **GPU T4 x1**, Internet **On**. **Save Version -> Save & Run All**.
Expected time: about 2-2.5 hours. Output: `satquery_smolvlm.zip`.

In [ ]:
# 1. Code (generalize branch) and libraries
!nvidia-smi -L
!rm -rf /tmp/SatQuery && git clone -q --depth 1 -b main https://github.com/Vansh-Vaid/SatQuery.git /tmp/SatQuery
!cd /tmp/SatQuery && git log --oneline -1
!pip install -q -U "transformers>=4.50" peft accelerate rasterio 2>&1 | tail -1

In [ ]:
# 2. VRSBench (about 12 GB) into /tmp
import os, time
from huggingface_hub import hf_hub_download
ROOT = "/tmp/vrsbench"
os.makedirs(ROOT, exist_ok=True)
t0 = time.time()
for f in ["VRSBench_train.json", "VRSBench_EVAL_vqa.json", "VRSBench_EVAL_Cap.json", "Images_val.zip", "Images_train.zip"]:
    for attempt in range(3):
        try:
            hf_hub_download("xiang709/VRSBench", f, repo_type="dataset", local_dir=ROOT)
            break
        except Exception as e:
            print(f"{f}: attempt {attempt + 1} failed ({e}); retrying")
            time.sleep(10)
    else:
        raise RuntimeError(f"could not download {f}; is Internet switched on in Settings?")
    print(f"{f}: {os.path.getsize(os.path.join(ROOT, f)) / 1e9:.2f} GB ({(time.time() - t0) / 60:.1f} min)")

In [ ]:
%%bash
set -eo pipefail
# 3. Chat rows with the app's prompts (12,000 training rows; held-out evaluation rows)
cd /tmp/SatQuery && python training/prepare_vrsbench_vlm.py --root /tmp/vrsbench --out /tmp/vrs_vlm


In [ ]:
%%bash
set -eo pipefail
# 4. LoRA fine-tune (stops itself inside the time limit, with the learning-rate schedule re-fitted)
cd /tmp/SatQuery && python training/finetune_vlm.py --data /tmp/vrs_vlm --out /kaggle/working/smolvlm-rs-lora \
    --base HuggingFaceTB/SmolVLM-256M-Instruct --epochs 1 --lr 3e-4 --rank 16 --alpha 32 --grad-accum 8 --val-limit 100 --eval-every 200 \
    --time-limit-min 150


In [ ]:
%%bash
set -eo pipefail
# 5. Score the untrained and the adapted model on the same held-out rows (and time answers on the CPU)
cd /tmp/SatQuery && python eval/score_small_vlm.py --data /tmp/vrs_vlm --base HuggingFaceTB/SmolVLM-256M-Instruct --out /kaggle/working/score_base.json --cpu-rows 4
cd /tmp/SatQuery && python eval/score_small_vlm.py --data /tmp/vrs_vlm --base HuggingFaceTB/SmolVLM-256M-Instruct --adapter /kaggle/working/smolvlm-rs-lora --out /kaggle/working/score_lora.json --cpu-rows 6


In [ ]:
# 6. Verdict and packing
import json, os, zipfile
b, l = (json.load(open(f"/kaggle/working/score_{k}.json")) for k in ("base", "lora"))
for key in ("vqa_accuracy", "caption_bleu4", "caption_rougeL", "cpu_seconds_per_answer"):
    print(f"{key:24s} untrained {b.get(key)!s:>8}   adapted {l.get(key)!s:>8}")
better = l["vqa_accuracy"] > b["vqa_accuracy"] and l["caption_rougeL"] >= b["caption_rougeL"]
fast = (l.get("cpu_seconds_per_answer") or 99) <= 15
print("\nVERDICT:", "BETTER and fast enough for the CPU server" if better and fast else
      ("BETTER but too slow on a CPU" if better else "not better than the untrained model"))
with zipfile.ZipFile("/kaggle/working/satquery_smolvlm.zip", "w") as z:
    for root, _, files in os.walk("/kaggle/working/smolvlm-rs-lora"):
        for f in files:
            p = os.path.join(root, f)
            z.write(p, os.path.relpath(p, "/kaggle/working"))
    for k in ("base", "lora"):
        z.write(f"/kaggle/working/score_{k}.json", f"score_{k}.json")
print("Download satquery_smolvlm.zip from the Output tab.")

## Afterwards
Unzip into `checkpoints/`. To serve it: `SATQUERY_VLM_BACKEND=hf SATQUERY_VLM_BASE=HuggingFaceTB/SmolVLM-256M-Instruct
SATQUERY_VLM_ADAPTER=checkpoints/smolvlm-rs-lora`. It goes live only if the verdict above is BETTER and fast enough.